In [1]:
from datetime import datetime
from pathlib import Path
import re

import numpy as np
import pandas as pd

from utils import write_csv
from utils import display_excel_sheets

from models import (
    DataSource,
    Population
)

def extract_single_country(text):
    pattern = r"\b(England|Scotland|Wales|Northern Ireland)\b"
    countries = re.findall(pattern, text)
    if len(countries) == 1:
        return countries[0]
    return None  



In [10]:
# define  datasource and publisher at top
# datasource used to make data directory
datasource_name = "ons-ukpopulationestimates"
publisher = "ons"


# load data

data_dir = (
    #Path(__file__).resolve().parent.parent.parent 
    Path.cwd().resolve().parent.parent / "data" / f"{datasource_name}"
)
output_dir = data_dir / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
# ------------------------------------------
# DataSource table
# ------------------------------------------
# https://unfccc.int/topics/mitigation/resources/registry-and-data/ghg-data-from-unfccc
datasource_name = "ons-ukpopulationestimates"
publisher = "ons"


datasource = DataSource(
    id=f"{publisher}:{datasource_name}",
    name="Population estimates for the UK and constituent countries by sex and age",
    publisher=publisher,
    published_date=datetime.strptime("2024-07-15", "%Y-%m-%d"),
    version="mid-2023",
    url=f"https://www.ons.gov.uk/file?uri=/peoplepopulationandcommunity/populationandmigration/populationestimates/datasets/populationestimatesforukenglandandwalesscotlandandnorthernireland/mid2024/mye24tablesuk.xlsx",
)

In [4]:
# ------------------------------------------
# get actors
# ------------------------------------------

fl_actors = (
    #Path(__file__).resolve().parent.parent.parent
    Path.cwd().resolve().parent.parent
    / "data"
    / "iso-3166-2"
    / "raw"
    / "subdivisions.csv"
)

df_actors = (
    pd.read_csv(fl_actors, keep_default_na=False)
    .loc[:, ["#country_code_alpha2",	"subdivision_code_iso3166-2",	"subdivision_name"]]
    .rename(columns={"#country_code_alpha2": "country_code_alpha2"})
    .query("country_code_alpha2 == 'GB'")
    #.rename(columns={"#country_code_alpha2":"alpha2", "country_code_alpha3":"alpha3"})
    )

In [22]:

fl = (
    #Path(__file__).resolve().parent.parent.parent
    Path.cwd().resolve().parent.parent
    / "data"
    / "gb-subnational-population"
    / "raw"
    / "ukpopulationestimates183820231.xlsx"
)

#fl = "/Users/lukegloege/projects/ghg-tracker-database/data/gb-subnational-population/raw/ukpopulationestimates183820231.xlsx"

#display_excel_sheets(fl)

list_of_data = []

for table in [10, 12, 14, 17]:
    #table=10
    sheet_name = f"Table {table}"
    header = pd.read_excel(fl, sheet_name=sheet_name, nrows=1).columns[0]
    subnational = extract_single_country(header)

    actor_id = df_actors.query("subdivision_name == @subnational").loc[:, "subdivision_code_iso3166-2"].iat[0]

    df_pop = (
        pd.read_excel(fl, sheet_name=sheet_name, skiprows=1)
        .assign(
            year = lambda d: d["Year"].apply(lambda x: x.split("-")[1]),
            actor_id = actor_id, 
            #id = lambda d: d.apply(lambda x: f"{x['actor_id']}:{x['year']}", axis=1),
            #datasource_id = datasource.id
            )
        .rename(columns={"Persons":"population"})
        .query("population != 'missing data'")
        .loc[:, ["actor_id", "year", "population"]]
        .sort_values(by="year")
        )
    
    list_of_data.append(df_pop)

    df = pd.concat(d for d in list_of_data)


df = df.astype({
    "actor_id": str,
    "year": int,
    "population": int,
})


In [7]:
df.sample(20)
# DONE

,id,actor_id,year,population,datasource_id
78,GB-NIR:1944,GB-NIR,1944,1357000,ons:ons-ukpopulationestimates
27,GB-SCT:1995,GB-SCT,1995,5103690,ons:ons-ukpopulationestimates
13,GB-SCT:2009,GB-SCT,2009,5231900,ons:ons-ukpopulationestimates
39,GB-NIR:1983,GB-NIR,1983,1550560,ons:ons-ukpopulationestimates
43,GB-NIR:1979,GB-NIR,1979,1528253,ons:ons-ukpopulationestimates
21,GB-SCT:2001,GB-SCT,2001,5064200,ons:ons-ukpopulationestimates
4,GB-ENG:2019,GB-ENG,2019,56230056,ons:ons-ukpopulationestimates
88,GB-SCT:1934,GB-SCT,1934,4934000,ons:ons-ukpopulationestimates
124,GB-SCT:1898,GB-SCT,1898,4344589,ons:ons-ukpopulationestimates
38,GB-ENG:1985,GB-ENG,1985,47057359,ons:ons-ukpopulationestimates


In [8]:
df.head()

,id,actor_id,year,population,datasource_id
52,GB-ENG:1971,GB-ENG,1971,46411700,ons:ons-ukpopulationestimates
51,GB-ENG:1972,GB-ENG,1972,46571900,ons:ons-ukpopulationestimates
50,GB-ENG:1973,GB-ENG,1973,46686200,ons:ons-ukpopulationestimates
49,GB-ENG:1974,GB-ENG,1974,46682700,ons:ons-ukpopulationestimates
48,GB-ENG:1975,GB-ENG,1975,46674400,ons:ons-ukpopulationestimates


In [23]:

populations_validated = [
    Population(
        id=f"{row.actor_id}:{row.year}",
        year=row.year,
        actor_id=row.actor_id,
        population=row.population,
        datasource_id=datasource.id,
    ).model_dump()
    for row in df.itertuples()
]



In [24]:
write_csv(
    output_dir=output_dir,
    name=Population.__tablename__,
    data=populations_validated,
    mode="w",
)
